<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# 🤖 Prédiction de tickets à risque — Machine Learning

## 🎯 Objectif

Dans ce notebook, nous allons construire un modèle de Machine Learning capable de **détecter automatiquement les tickets à risque** dès leur création :

- Ticket risquant de **dépasser son SLA**
- Ticket risquant de **partir en backlog**
- Ticket risquant de provoquer une **escalade**

---

## 💡 Contexte Business

Un service support gère des milliers de tickets par mois.

Il est impossible de surveiller manuellement chaque ticket.

Nous allons donc :
- définir la variable cible (ticket_at_risk)
- construire des features disponibles dès la création du ticket
- entraîner un modèle ML
- optimiser le seuil de détection selon une contrainte métier
- générer un fichier d'alertes pour Power BI

---

### Prérequis
- Fichier `support_clean_analytics.csv` produit par le Notebook 2 :
  - Colab : /content/drive/MyDrive/DataProjectLab/projects/customer_support_analytics/
  - Local : ./outputs/

## 🧩 Données utilisées

- `support_clean_analytics.csv` (généré en Notebook 2)
- Colonne cible : `ticket_at_risk` (1 = à risque, 0 = normal)

---

## 🧠 Approche

1. Chargement et exploration du dataset ML
2. Coupure temporelle train/test (règle absolue : pas de data leakage)
3. Entraînement de 3 modèles
4. Évaluation et comparaison
5. Feature importance
6. Courbes ROC et optimisation du seuil
7. Génération du fichier d'alertes pour Power BI
8. Validation croisée temporelle

---

##### 🔧 1. Imports

In [2]:
# ============================================================
# 1. IMPORTS
# ============================================================
import re
import os, sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              roc_curve, precision_recall_curve, f1_score,
                              precision_score, recall_score, accuracy_score)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

COLORS = {
    "primary": "#534AB7", "secondary": "#1D9E75",
    "warning": "#EF9F27", "danger": "#E24B4A", "neutral": "#888780"
}

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/DataProjectLab/customer_support_analytics/"
else:
    SAVE_PATH = "./outputs/"
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"📁 Environnement : {'Colab' if IN_COLAB else 'Local'}")
print(f"📁 Dossier       : {SAVE_PATH}") 

print("✅ Librairies ML chargées")
print("💡 Note : pip install shap  si vous souhaitez utiliser SHAP pour l'interprétabilité")

📁 Environnement : Local
📁 Dossier       : ./outputs/
✅ Librairies ML chargées
💡 Note : pip install shap  si vous souhaitez utiliser SHAP pour l'interprétabilité


---

##### 📥 2. Chargement des données

Nous chargeons le dataset propre généré en Notebook 2.

> ⚠️ Si `support_clean_analytics.csv` n'a pas encore été généré, exécute d'abord le Notebook 2.

In [3]:
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_support_analytics/data/"
clean_path = f"{SAVE_PATH}support_clean_analytics.csv"

# Vérifier si le fichier nettoyé existe (produit par le NB2)
# Sinon, fallback vers les données brutes GitHub
if os.path.exists(clean_path):
    file_source = clean_path
    print(f"✅ Fichier nettoyé trouvé : {clean_path}")
else:
    file_source = BASE_URL + "support_clean_analytics.csv"
    print(f"⚠️  Fichier nettoyé non trouvé — chargement depuis GitHub")
    print(f"   → Exécute le Notebook 2 pour produire le fichier nettoyé")

df = pd.read_csv(f"{file_source}", parse_dates=["created_at"])

print(df.shape)
display(df.head())

⚠️  Fichier nettoyé non trouvé — chargement depuis GitHub
   → Exécute le Notebook 2 pour produire le fichier nettoyé
(15287, 37)


,ticket_id,created_at,category_id,agent_id,pays,canal,priorite,statut,sla_heures,first_response_heures,...,sla_strict,heure_creation,jour_semaine,mois,est_weekend,heure_hors_bureau,ticket_at_risk,is_high_priority,is_long_wait,is_reopened
0,TKT000001,2022-01-01 08:06:00,CAT006,AGT001,Ghana,Email,3,Backlog,360,29.1,...,False,8,5,1,1,0,1,0,1,0
1,TKT000002,2022-01-01 13:45:00,CAT001,AGT004,CI,Téléphone,4,Résolu,240,56.2,...,True,13,5,1,1,0,1,0,1,0
2,TKT000003,2022-01-01 08:48:00,CAT008,AGT003,Maroc,Chat,2,Résolu,120,5.0,...,True,8,5,1,1,0,0,1,1,0
3,TKT000004,2022-01-01 19:05:00,CAT007,AGT012,France,Email,5,En cours,480,139.5,...,False,19,5,1,1,1,1,0,1,0
4,TKT000005,2022-01-01 22:54:00,CAT007,AGT001,Maroc,Téléphone,5,Escaladé,480,68.4,...,False,22,5,1,1,1,1,0,1,1


---

##### 🔍 3. Analyse initiale

Vérifie :
- les types de données
- la distribution de la variable cible `ticket_at_risk`
- le ratio de déséquilibre entre les deux classes

In [4]:
print(df.dtypes)
print(df.isnull().sum())

print("\nDistribution de la variable cible :")
print(df["ticket_at_risk"].value_counts())
print(f"\nTaux de tickets à risque : {df['ticket_at_risk'].mean()*100:.1f}%")
print(f"Ratio déséquilibre : 1 risque pour {(df['ticket_at_risk']==0).sum()//df['ticket_at_risk'].sum()} normaux")

ticket_id                        object
created_at               datetime64[ns]
category_id                      object
agent_id                         object
pays                             object
canal                            object
priorite                          int64
statut                           object
sla_heures                        int64
first_response_heures           float64
resolution_heures               float64
sla_breach                        int64
nb_contacts                       int64
reopened                          int64
csat                            float64
in_backlog                        int64
ratio_sla                       float64
nom                              object
tier                             object
tier_num                          int64
bureau                           object
csat_moyen                      float64
taux_resolution_pct               int64
tickets_actifs                    int64
nom_cat                          object


### 🧠 Analyse

- Le dataset est-il déséquilibré ?
- Quelle technique utiliseras-tu pour gérer ce déséquilibre ?

> 💡 Si la classe 1 représente < 40% : utilise `class_weight='balanced'` dans Random Forest et Logistic Regression.  
> Pour Gradient Boosting : utilise `compute_sample_weight('balanced', y_train)`.

*(rédige ici)*

---

##### 🎯 4. Définition des features

Définis la liste des features à utiliser.

> ⚠️ **Règle absolue — anti-leakage :**  
> N'utilise **jamais** `resolution_heures`, `sla_breach`, `csat`, `ratio_sla`, `in_backlog` comme feature.  
> Ces valeurs ne sont connues qu'**après** la résolution du ticket.  
> En production, le modèle prédit **au moment de la création** du ticket.

Features disponibles à la création du ticket :
- `priorite`, `sla_heures`, `canal_enc`, `mois`, `jour_semaine`, `heure_creation`
- `est_weekend`, `heure_hors_bureau`
- `tier_num`, `csat_moyen`, `taux_resolution_pct`, `tickets_actifs`
- `sla_strict`, `taux_resolution_historique`, `domaine_enc`, `priorite_defaut`

In [5]:
FEATURES = [
    "priorite",
    "sla_heures",
    "canal_enc",
    "mois",
    "jour_semaine",
    "heure_creation",
    "est_weekend",
    "heure_hors_bureau",
    "tier_num",
    "csat_moyen",
    "taux_resolution_pct",
    "tickets_actifs",
    "sla_strict",
    "taux_resolution_historique",
    "domaine_enc",
    "priorite_defaut"
]

TARGET = "ticket_at_risk"

# Encodage si les colonnes ne sont pas encore encodées
# Votre code ici

# Vérifier que toutes les features existent
FEATURES = [f for f in FEATURES if f in df.columns]
print(f"✅ Features retenues : {len(FEATURES)}")
print(FEATURES)

✅ Features retenues : 13
['priorite', 'sla_heures', 'mois', 'jour_semaine', 'heure_creation', 'est_weekend', 'heure_hors_bureau', 'tier_num', 'csat_moyen', 'taux_resolution_pct', 'tickets_actifs', 'sla_strict', 'priorite_defaut']


---

##### 🔀 5. Coupure temporelle train/test

**Règle absolue pour les données de support :**  
Trie par `created_at` et coupe **sans mélange**.

- **Train** : tickets créés avant le 2024-01-01
- **Test** : tickets créés en 2024

> ⚠️ **Pourquoi pas `train_test_split` aléatoire ?**  
> En production, le modèle prédit sur des tickets **futurs**.  
> Un split aléatoire laisserait entrer des patterns futurs dans le train — c'est du data leakage temporel.

In [6]:
# Coupure temporelle stricte
# Votre code ici

train =
test  =

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train : {len(X_train):,} obs | Risk rate : {y_train.mean()*100:.1f}%")
print(f"Test  : {len(X_test):,}  obs | Risk rate : {y_test.mean()*100:.1f}%")

SyntaxError: invalid syntax (2798982744.py, line 4)

---

##### 🤖 6. Entraînement de 3 modèles

Entraîne et compare :

**Modèle 1 — Logistic Regression** (baseline, très interprétable)
- `class_weight='balanced'`, `C=1.0`, `max_iter=500`

**Modèle 2 — Random Forest** (robuste, feature importance native)
- `n_estimators=300`, `max_depth=8`, `class_weight='balanced'`, `random_state=42`

**Modèle 3 — Gradient Boosting** (souvent le plus performant sur données tabulaires)
- `n_estimators=200`, `max_depth=4`, `learning_rate=0.1`, `subsample=0.8`
- ⚠️ Utilise `compute_sample_weight('balanced', y_train)` car GBM n'a pas `class_weight`

Pour chaque modèle :
- Calcule : Accuracy, Precision, Recall, F1-Score (classe 1), AUC-ROC
- Affiche la matrice de confusion (`sns.heatmap`)

Tableau comparatif final des 3 modèles.

In [ ]:
sw = compute_sample_weight("balanced", y_train)

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", C=1.0, max_iter=500),
    "Random Forest":       RandomForestClassifier(n_estimators=300, max_depth=8,
                                                   class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                                       learning_rate=0.1, subsample=0.8, random_state=42),
}

results = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for (name, model), ax in zip(models.items(), axes):
    # Entraînement
    # Votre code ici

    # Prédictions
    # Votre code ici

    # Métriques
    # Votre code ici

    # Matrice de confusion
    # Votre code ici

plt.tight_layout()
plt.show()

# Tableau comparatif
print("\nCOMPARAISON MODÈLES :")
print(f"{'Modèle':<25} {'AUC':>8} {'F1':>8} {'Recall':>8}")
for nm, r in results.items():
    print(f"{nm:<25} {r['AUC']:>8} {r['F1']:>8} {r['Recall']:>8}")

### 🧠 Analyse des résultats

- Quel modèle a le meilleur AUC-ROC ?
- Quel modèle a le meilleur Recall sur la classe "à risque" ?
- Pourquoi optimise-t-on le Recall plutôt que la Precision dans ce contexte ?

*(rédige ici)*

---

##### 📊 7. Feature Importance

**Pour le meilleur modèle :**

1. Extrais et visualise les 15 features les plus importantes (barres horizontales)
2. Groupe les features par catégorie :
   - Features temporelles : `mois`, `jour_semaine`, `heure_creation`, `est_weekend`, `heure_hors_bureau`
   - Features agent : `tier_num`, `csat_moyen`, `taux_resolution_pct`, `tickets_actifs`
   - Features catégorie : `sla_strict`, `taux_resolution_historique`, `domaine_enc`, `priorite_defaut`
   - Features ticket : `priorite`, `sla_heures`, `canal_enc`

3. Calcule l'importance totale par groupe

Trace le graphique côte à côte : importance individuelle (gauche) + par groupe (droite).

> 💡 `rf.feature_importances_` pour Random Forest  
> `model.coef_` pour Logistic Regression

In [ ]:
# Feature Importance du meilleur modèle
# Votre code ici

### 🧠 Interprétation métier

Que signifient les features les plus importantes pour M. KOAUME ?  
Traductions en actions opérationnelles possibles ?

*(rédige ici)*

---

##### 📈 8. Courbes ROC et optimisation du seuil

La contrainte métier de M. KOAUME :  
> *"Je veux que mon système d'alerte capture au moins 75% des tickets vraiment à risque, même si ça génère quelques fausses alertes."*

**Ce que tu dois faire :**

1. Trace les 3 courbes ROC sur le même graphique
2. Trace la courbe Precision-Recall
3. Compare 6 seuils : 0.25, 0.30, 0.35, 0.40, 0.45, 0.50 pour le meilleur modèle  
   - Pour chaque seuil : Precision · Recall · F1 · % de tickets "alertés"
4. Identifie le seuil qui respecte **Recall > 0.75** avec le moins de fausses alertes

> 💡 `y_prob = model.predict_proba(X_test)[:, 1]`  
> Seuil personnalisé : `y_pred_custom = (y_prob >= seuil).astype(int)`

In [ ]:
# Courbes ROC comparatives
# Votre code ici

In [ ]:
# Courbe Precision-Recall
# Votre code ici

In [ ]:
# Comparaison des seuils
print("COMPARAISON DES SEUILS")
print(f"{'Seuil':<8} {'Precision':>10} {'Recall':>10} {'F1':>8} {'Alertes%':>10}")

# Votre code ici

### 🧠 Recommandation de seuil

Quel seuil recommandes-tu ? Justifie avec les chiffres.

*(rédige ici)*

---

##### 🧾 9. Génération du fichier d'alertes pour Power BI

Génère `tickets_risque_scores.csv` avec pour chaque ticket du test set :
- `ticket_id`, `created_at`, `category_id`, `agent_id`, `pays`, `canal`, `priorite`, `statut`
- `score_risque` : probabilité prédite (0 à 1)
- `niveau_alerte` :
  - Score > 0.70 → "ROUGE — Intervention immédiate"
  - Score > 0.45 → "ORANGE — Surveiller"
  - Score > 0.25 → "JAUNE — Attention"
  - Sinon        → "VERT — Normal"
- `ticket_at_risk` : valeur réelle (pour validation dans Power BI)

In [ ]:
def niveau_alerte(score):
    if score > 0.70:
        return "ROUGE — Intervention immédiate"
    if score > 0.45:
        return "ORANGE — Surveiller"
    if score > 0.25:
        return "JAUNE — Attention"
    return "VERT — Normal"

# Application des prédictions et génération du fichier
# Votre code ici

df_alertes.to_csv(f'{SAVE_PATH}tickets_risque_scores.csv', index=False, encoding="utf-8-sig")

print("✅ Fichier tickets_risque_scores.csv généré")
print("\nDistribution des niveaux :")
print(df_alertes["niveau_alerte"].value_counts())

---

##### 🧪 10. Validation croisée temporelle

Valide le modèle de façon réaliste avec `TimeSeriesSplit` (5 folds temporels).

> 💡 `TimeSeriesSplit` garantit que le fold N est toujours entraîné sur des données **antérieures** au fold N+1.  
> C'est la seule validation croisée valide pour des séries temporelles.

Compare les résultats avec le split simple train/test du point 5.

> `from sklearn.model_selection import TimeSeriesSplit`  
> `tscv = TimeSeriesSplit(n_splits=5)`  
> `scores = cross_val_score(rf, X, y, cv=tscv, scoring='roc_auc')`

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Validation croisée temporelle
# Votre code ici

# Visualisation des scores par fold
# Votre code ici

### 🧠 Stabilité du modèle

- Le score AUC est-il stable entre les folds ?
- Y a-t-il une dégradation sur les folds les plus récents ?

*(rédige ici)*

---

##### 🧠 11. Conclusion

## 🧠 Conclusion

**Ce que tu as réalisé :**

✔ Chargé et exploré le dataset ML  
✔ Appliqué une coupure temporelle stricte (anti-leakage)  
✔ Entraîné et comparé 3 modèles  
✔ Analysé la feature importance  
✔ Optimisé le seuil selon la contrainte Recall > 0.75  
✔ Généré le fichier d'alertes pour Power BI  
✔ Validé avec TimeSeriesSplit  

---

## 🚀 Utilisation Business

Ce modèle permet de :
- alerter les superviseurs en temps réel sur les tickets à risque
- prioriser le traitement des tickets critiques
- anticiper les SLA breach avant qu'ils se produisent
- réduire le backlog en intervenant plus tôt

---

## ✏️ Ta recommandation finale

Avec les métriques obtenues, le modèle est-il prêt pour la production ?  
Quelles améliorations proposerais-tu (plus de données, features supplémentaires, modèle différent) ?

*(rédige ici)*

---

👉 **Intègre `tickets_risque_scores.csv` dans Power BI — Page Alertes (Notebook 4)**

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.